# Problema: Previsão de atrasos de avião

Os objetivos deste bloco de anotações são:
- Processar e criar um conjunto de dados com base em arquivos .zip obtidos por download
- Executar análise exploratória de dados (AED)
- Estabelecer um modelo de linha de base
- Ir de um modelo simples para um modelo de conjunto
- Executar a otimização de hiperparâmetros
- Verificar a importância do recurso


## Introdução ao cenário do negócio

Você trabalha para um site de reservas de viagens que deseja melhorar a experiência do cliente referente a voos atrasados. A empresa deseja criar um recurso para informar aos clientes se o voo sofrerá atraso devido a condições meteorológicas, em reservas que usarão os aeroportos mais movimentados para viagens domésticas nos EUA. 

Sua tarefa será resolver parte desse problema usando machine learning (ML) para identificar se o voo sofrerá atraso devido a condições meteorológicas. Você recebeu acesso ao conjunto de dados sobre a pontualidade dos voos domésticos operados por grandes companhias aéreas. Use esses dados no treinamento de um modelo de ML para prever se o voo sofrerá atraso nos aeroportos mais movimentados.


## Sobre esse conjunto de dados

Esse conjunto de dados contém os horários de partidas e chegadas previstos e reais relatados pelas companhias aéreas certificadas dos EUA, que representam pelo menos 1 por cento da receita prevista de passageiros domésticos. Os dados foram coletados pelo U.S. Office of Airline Information, Bureau of Transportation Statistics (BTS). O conjunto de dados contém data, hora, origem, destino, companhia aérea, distância e status de atraso dos voos entre 2013 e 2018.


### Componentes (features)

Para obter mais informações sobre os recursos do conjunto de dados, consulte [Recursos do conjunto de dados de pontualidade/atraso] (https://www.transtats.bts.gov/Fields.asp).

### Atribuições do conjunto de dados  
Site: https://www.transtats.bts.gov/

Os conjuntos de dados usados neste laboratório foram compilados pelo U.S. Office of Airline Information, Bureau of Transportation Statistics (BTS), Airline On-Time Performance Data, disponível em https://www.transtats.bts.gov/DatabaseInfo.asp?DB_ID=120&amp;DB_URL=Mode_ID=1&amp;Mode_Desc=Aviation&amp;Subject_ID2=0.

# Etapa 1: Formulação do problema e coleta de dados

Inicie esse projeto escrevendo algumas frases que resumem o problema do negócio e o objetivo que você deseja alcançar neste cenário. Você pode anotar suas ideias nas seções a seguir. Inclua uma métrica de negócios à qual você gostaria que sua equipe aspirasse. Depois de definir essas informações, escreva a declaração do problema de ML. Por fim, adicione um ou dois comentários sobre o tipo de ML que essa atividade representa. 

#### <span style="color: blue;">Apresentação do projeto: inclua um resumo desses detalhes na apresentação do projeto.</span>

### 1. Determine se e por que o ML é uma solução apropriada para implantação nesse cenário.

In [ ]:
# Write your answer here

### 2. Formule o problema do negócio, as métricas de sucesso e o resultado de ML desejado.

In [ ]:
# Write your answer here

### 3. Identifique o tipo de problema de ML com o qual você está trabalhando.

In [ ]:
# Write your answer here

### 4. Analise a adequação dos dados com os quais você está trabalhando.

In [ ]:
# Write your answer here

### Configuração

Agora que você decidiu onde deseja concentrar sua atenção, configure este laboratório para que possa começar a resolver o problema.

** Observação:** Este bloco de anotações foi criado e testado em uma instância de bloco de anotações `ml.m4.xlarge` com 25 GB de armazenamento. 

In [ ]:
import os
from pathlib2 import Path
from zipfile import ZipFile
import time

import pandas as pd
import numpy as np
import subprocess

import matplotlib.pyplot as plt
import seaborn as sns

sns.set()
instance_type='ml.m4.xlarge'

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Etapa 2: Pré-processamento e visualização dos dados  
Nesta fase de pré-processamento de dados, você explorará e visualizará seus dados para entendê-los melhor. Primeiramente, importe as bibliotecas necessárias e leia os dados em um DataFrame do pandas. Depois de importar os dados, explore o conjunto de dados. Procure a forma do conjunto de dados e explore suas colunas e os tipos de colunas com os quais você trabalhará (numéricas, categóricas). Considere a hipótese de executar análises estatísticas básicas dos recursos para ter uma ideia dos recursos e dos intervalos. Examine atentamente sua coluna de destino e determine sua distribuição.


### Perguntas específicas a considerar

Durante esta seção do laboratório, considere as seguintes perguntas:

1. O que você pode deduzir das análises estatísticas básicas que executou nos recursos? 
2. O que você pode deduzir das distribuições das classes de destino?
3. Há mais alguma coisa que você possa deduzir ao explorar os dados?

#### <span style="color: blue;">Apresentação do projeto: inclua um resumo das suas respostas a essas perguntas (e a outras perguntas semelhantes) na apresentação do projeto.</span>

Comece trazendo o conjunto de dados de um bucket público do Amazon Simple Storage Service (Amazon S3) para o ambiente deste bloco de anotações.

In [ ]:
# download the files

zip_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
base_path = '/home/ec2-user/SageMaker/project/data/FlightDelays/'
csv_base_path = '/home/ec2-user/SageMaker/project/data/csvFlightDelays/'

!mkdir -p {zip_path}
!mkdir -p {csv_base_path}
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data/ {zip_path} --recursive


In [ ]:
zip_files = [str(file) for file in list(Path(base_path).iterdir()) if '.zip' in str(file)]
len(zip_files)

Extraia arquivos de valores separados por vírgula (CSV) dos arquivos .zip.

In [ ]:
def zip2csv(zipFile_name , file_path):
    """
    Extract csv from zip files
    zipFile_name: name of the zip file
    file_path : name of the folder to store csv
    """

    try:
        with ZipFile(zipFile_name, 'r') as z: 
            print(f'Extracting {zipFile_name} ') 
            z.extractall(path=file_path) 
    except:
        print(f'zip2csv failed for {zipFile_name}')

for file in zip_files:
    zip2csv(file, csv_base_path)

print("Files Extracted")

In [ ]:
csv_files = [str(file) for file in list(Path(csv_base_path).iterdir()) if '.csv' in str(file)]
len(csv_files)

Antes de carregar o arquivo CSV, leia o arquivo HTML da pasta extraída. Esse arquivo HTML inclui o histórico e mais informações sobre os recursos incluídos no conjunto de dados.

In [ ]:
from IPython.display import IFrame

IFrame(src=os.path.relpath(f"{csv_base_path}readme.html"), width=1000, height=600)

#### Carregar o arquivo CSV de exemplo

Antes de combinar todos os arquivos CSV, examine os dados de um único arquivo CSV. Usando o pandas, leia primeiramente o arquivo `On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv`. Você pode usar a função `read_csv` integrada em Python ([pandas.read_csv documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)).

In [ ]:
df_temp = pd.read_csv(f"{csv_base_path}On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2018_9.csv")

**Pergunta**: Imprima o comprimento da linha e da coluna no conjunto de dados e imprima os nomes das colunas.

**Dica**: Para visualizar as linhas e colunas de um DataFrame, use a função `<DataFrame>.shape` function. To view the column names, use the `<DataFrame>.columns`.

In [ ]:
df_shape = # **ENTER YOUR CODE HERE**
print(f'Rows and columns in one CSV file is {df_shape}')

**Pergunta**: Imprima as primeiras 10 linhas do conjunto de dados.  

** Dica**: Para imprimir o número `x` de linhas, use a função `head(x)` integrada no pandas.

In [ ]:
# Enter your code here

**Pergunta**: Imprima todas as colunas do conjunto de dados. Para visualizar os nomes das colunas, use `<DataFrame>.columns`.

In [ ]:
print(f'The column names are :')
print('#########')
for col in <CODE>:# **ENTER YOUR CODE HERE**
    print(col)

**Pergunta**: Imprima todas as colunas do conjunto de dados que contenham a palavra *Del*. Isso ajudará você a ver quantas colunas possuem *delay data* (dados de atraso) nelas.

**Dica**: Para incluir valores que passam por determinados critérios de declaração de `if`, use uma compreensão de lista do Python.

Por exemplo: `[x for x in [1,2,3,4,5] if x > 2]`  

**Dica**: Para verificar se o valor está em uma lista, você pode usar a palavra-chave `in` ([Python na documentação de palavras-chave] (https://www.w3schools.com/python/ref_keyword_in.asp)). 

Por exemplo: `5 in [1,2,3,4,5]`

In [ ]:
# Enter your code here

Aqui estão mais algumas perguntas para ajudar você a saber mais sobre seu conjunto de dados.

**Perguntas**   

1. Quantas linhas e colunas o conjunto de dados possui?   
2. Quantos anos estão incluídos no conjunto de dados?   
3. Qual é o intervalo de datas do conjunto de dados?   
4. Que companhias aéreas estão incluídas no conjunto de dados?   
5. Que aeroportos de origem e destino estão cobertos?

**Dicas**
- Para mostrar as dimensões do DataFrame, use `df_temp.shape`.
- Para fazer referência a uma coluna específica, use `df_temp.columnName` (por exemplo, `df_temp.CarrierDelay`).
- Para obter valores exclusivos para uma coluna, use `df_temp.column.unique()` (por exemplo, `df_temp.Year.unique()`).

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", <CODE>)
print("The months covered in this dataset are: ", <CODE>)
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

**Pergunta**: Qual é a contagem de todos os aeroportos de origem e destino?

**Dica**: Para encontrar os valores de cada aeroporto usando as colunas **Origin** (Origem) e **Dest** (Destino), use a função `values_count` no pandas ([pandas.Series.value_counts documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)).

In [ ]:
counts = pd.DataFrame({'Origin':<CODE>, 'Destination':<CODE>})
counts

**Pergunta**: Imprima os 15 principais aeroportos de origem e destino com base no número de voos incluídos no conjunto de dados.

**Dica**: Você pode usar a função `sort_values` no pandas ([pandas.DataFrame.sort_values documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_values.html)).

In [ ]:
counts.sort_values(by=<CODE>,ascending=False).head(15) # Enter your code here

**Considerando todas as informações sobre um voo, você pode prever se ele atrasará?**

A coluna **ArrDel15** é uma variável indicadora que assume o valor *1* quando o atraso é superior a 15 minutos. Caso contrário, ela assume o valor *0*.

Você pode usar essa coluna como coluna de destino para o problema de classificação.

Agora, suponha que você esteja viajando de São Francisco para Los Angeles em uma viagem de trabalho. Você quer gerenciar melhor suas reservas em Los Angeles. Portanto, você deseja saber se o voo atrasará, com base em um conjunto de recursos. Quantos recursos desse conjunto de dados você precisa saber antes do voo?

Determinadas colunas, como `DepDelay`, `ArrDelay`, `CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay` e `DivArrDelay`, contêm informações sobre um atraso. Mas esse atraso pode ocorrer na origem ou no destino. Se houvesse um atraso repentino de 10 minutos devido a condições climáticas antes da aterrissagem, esses dados não seriam úteis para o gerenciamento das suas reservas em Los Angeles.

Portanto, para simplificar a declaração do problema, considere as seguintes colunas para prever um atraso na chegada:<br>

`Year`, `Quarter`, `Month`, `DayofMonth`, `DayOfWeek`, `FlightDate`, `Reporting_Airline`, `Origin`, `OriginState`, `Dest`, `DestState`, `CRSDepTime`, `DepDelayMinutes`, `DepartureDelayGroups`, `Cancelled`, `Diverted`, `Distance`, `DistanceGroup`, `ArrDelay`, `ArrDelayMinutes`, `ArrDel15`, `AirTime`

Você também filtrará os aeroportos de origem e destino como:
- Principais aeroportos: ATL, ORD, DFW, DEN, CLT, LAX, IAH, PHX, SFO
- As cinco principais companhias aéreas: UA, OO, WN, AA, DL

Essas informações deverão ajudar a reduzir o tamanho dos dados nos arquivos CSV que serão combinados.

#### Combinar todos os arquivos CSV
 
Primeiramente, crie um DataFrame vazio que você usará para copiar os DataFrames de cada arquivo. Em seguida, para cada arquivo na lista `csv_files`:

1. Leia o arquivo CSV em um quadro de dados 
2. Filtre as colunas com base na variável `filter_cols`

```
        columns = ['col1', 'col2']
        df_filter = df[columns]
```

3. Mantenha apenas o `subset_vals` em cada `subset_cols`. Para verificar se o `val` está na coluna DataFrame, use a função `isin` no pandas ([pandas.DataFram.isin documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html)). Em seguida, escolha as linhas que o incluem.

```
        df_eg[df_eg['col1'].isin('5')]
```

4. Concatenar o DataFrame com o DataFrame vazio 

In [ ]:
def combine_csv(csv_files, filter_cols, subset_cols, subset_vals, file_name):

    """
    Combine csv files into one Data Frame
    csv_files: list of csv file paths
    filter_cols: list of columns to filter
    subset_cols: list of columns to subset rows
    subset_vals: list of list of values to subset rows
    """

    df = pd.DataFrame()
    
    for file in csv_files:
        df_temp = pd.read_csv(file)
        df_temp = df_temp[filter_cols]
        for col, val in zip(subset_cols,subset_vals):
            df_temp = df_temp[df_temp[col].isin(val)]      
        
        df = pd.concat([df, df_temp], axis=0)
      
    df.to_csv(file_name, index=False)
    print(f'Combined csv stored at {file_name}')

In [ ]:
#cols is the list of columns to predict Arrival Delay 
cols = ['Year','Quarter','Month','DayofMonth','DayOfWeek','FlightDate',
        'Reporting_Airline','Origin','OriginState','Dest','DestState',
        'CRSDepTime','Cancelled','Diverted','Distance','DistanceGroup',
        'ArrDelay','ArrDelayMinutes','ArrDel15','AirTime']

subset_cols = ['Origin', 'Dest', 'Reporting_Airline']

# subset_vals is a list collection of the top origin and destination airports and top 5 airlines
subset_vals = [['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['ATL', 'ORD', 'DFW', 'DEN', 'CLT', 'LAX', 'IAH', 'PHX', 'SFO'], 
               ['UA', 'OO', 'WN', 'AA', 'DL']]

Use a função anterior para mesclar todos os arquivos diferentes em um único arquivo que você possa ler facilmente. 

**Observação**: Este processo demorará de 5 a 7 minutos para ser concluído.

In [ ]:
start = time.time()
combined_csv_filename = f"{base_path}combined_files.csv"
combine_csv(csv_files, cols, subset_cols, subset_vals, combined_csv_filename)
print(f'CSVs merged in {round((time.time() - start)/60,2)} minutes')

#### Fazer download do conjunto de dados

Carregue o conjunto de dados combinado.

In [ ]:
data = pd.read_csv(combined_csv_filename)

Imprima os primeiros cinco registros.

In [ ]:
# Enter your code here 

Aqui estão mais algumas perguntas para ajudar você a saber mais sobre seu conjunto de dados.

**Perguntas**   

1. Quantas linhas e colunas o conjunto de dados possui?   
2. Quantos anos estão incluídos no conjunto de dados?   
3. Qual é o intervalo de datas do conjunto de dados?   
4. Que companhias aéreas estão incluídas no conjunto de dados?   
5. Que aeroportos de origem e destino estão cobertos?

In [ ]:
print("The #rows and #columns are ", <CODE> , " and ", <CODE>)
print("The years in this dataset are: ", list(<CODE>))
print("The months covered in this dataset are: ", sorted(list(<CODE>)))
print("The date range for data is :" , min(<CODE>), " to ", max(<CODE>))
print("The airlines covered in this dataset are: ", list(<CODE>))
print("The Origin airports covered are: ", list(<CODE>))
print("The Destination airports covered are: ", list(<CODE>))

Defina a coluna de destino: **is_delay** (*1* significa que o horário de chegada atrasou mais de 15 minutos e *0* significa todos os outros casos). Para renomear a coluna de * ArrDel15** para *is_delay*, use o método `rename`.

** Dica**: Você pode usar a função `rename` no pandas ([pandas.DataFrame.rename documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

Por exemplo:
```
data.rename(columns={'col1':'column1'}, inplace=True)
```

In [ ]:
data.rename(columns=<CODE>, inplace=True) # Enter your code here

Procure nulos nas colunas. Você pode usar a função `isnull()` ([pandas.isnull documentation] (https://pandas.pydata.org/pandas-docs/version/0.17.0/generated/pandas.isnull.html)).

** Dica**: `isnull()` detecta se o valor em particular é nulo ou não. Ela retorna *True* (Verdadeiro) ou *False* (Falso) booleano em seu lugar. Para somar o número de colunas, use a função `sum(axis=0)` (por exemplo, `df.isnull().sum(axis=0)`).

In [ ]:
# Enter your code here

Os detalhes do atraso na chegada e o tempo de voo estão ausentes para 22.540 de 1.658.130 linhas, o que significa 1,3%. Você pode remover ou imputar essas linhas. A documentação não menciona informações sobre linhas ausentes.


In [ ]:
### Remove null columns
data = data[~data.is_delay.isnull()]
data.isnull().sum(axis = 0)

Obtenha a hora do dia no formato de 24 horas em CRSDepTime.

In [ ]:
data['DepHourofDay'] = (data['CRSDepTime']//100)

## **A declaração de problema de ML**
- Considerando um conjunto de recursos, você pode prever se um voo atrasará mais de 15 minutos?
- Como a variável de destino tem apenas um valor de *0* ou *1*, você pode usar um algoritmo de classificação. 

Antes de começar a modelar, é uma boa prática examinar a distribuição de recursos, as correlações e outros fatores.
- Isso lhe dará uma ideia de quaisquer não linearidades ou padrões nos dados
    - Modelos lineares: adicionar recursos de energia, exponenciais ou de interação
    - Experimente um modelo não linear
- Desequilíbrio de dados 
    - Escolha métricas que não fornecerão um desempenho de modelo tendencioso (precisão em comparação com a área sob a curva (AUC))
    - Use funções de perda ponderadas ou personalizadas
- Dados ausentes
    - Fazer imputação com base em estatísticas simples - média, mediana, modo (variáveis numéricas), classe frequente (variáveis categóricas)
    - Imputação baseada em clusterização (k-nearest neighbors, ou KNNs, para prever o valor da coluna)
    - Soltar coluna

### Exploração de dados

Verifique as classes *delay* (atraso) em comparação com *no delay* (sem atraso).


In [ ]:
(data.groupby('is_delay').size()/len(data) ).plot(kind='bar')# Enter your code here
plt.ylabel('Frequency')
plt.title('Distribution of classes')
plt.show()

**Pergunta**: O que você pode deduzir do gráfico de barras sobre a relação entre *delay* (atraso) e *no delay* (sem atraso)?

In [ ]:
# Enter your answer here

Execute as duas células a seguir e responda às perguntas.

In [ ]:
viz_columns = ['Month', 'DepHourofDay', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest']
fig, axes = plt.subplots(3, 2, figsize=(20,20), squeeze=False)
# fig.autofmt_xdate(rotation=90)

for idx, column in enumerate(viz_columns):
    ax = axes[idx//2, idx%2]
    temp = data.groupby(column)['is_delay'].value_counts(normalize=True).rename('percentage').\
    mul(100).reset_index().sort_values(column)
    sns.barplot(x=column, y="percentage", hue="is_delay", data=temp, ax=ax)
    plt.ylabel('% delay/no-delay')
    

plt.show()

In [ ]:
sns.lmplot( x="is_delay", y="Distance", data=data, fit_reg=False, hue='is_delay', legend=False)
plt.legend(loc='center')
plt.xlabel('is_delay')
plt.ylabel('Distance')
plt.show()

**Perguntas**

Usando os dados dos gráficos anteriores, responda a estas perguntas:

- Que meses apresentam mais atrasos?
- Que horário do dia apresenta mais atrasos?
- Que dia da semana apresenta mais atrasos?
- Que companhia aérea apresenta mais atrasos?
- Que aeroportos de origem e destino apresentam mais atrasos?
- A distância do voo é um fator nos atrasos?

In [ ]:
# Enter your answers here

### Componentes (features)

Examine todas as colunas e quais são seus tipos específicos.

In [ ]:
data.columns

In [ ]:
data.dtypes

Filtragem das colunas necessárias:
- *Date* (Data) é redundante, pois temos *Year* (Ano), *Quarter* (Trimestre), *Month* (Mês), *DayofMonth* (Dia do mês) e *DayOfWeek* (Dia da semana) para descrever a data.
- Use os códigos *Origin* (Origem) e *Dest* (Destino) em vez de *OriginState* (Estado de origem) e *DestState* (Estado de destino).
- Como você só está classificando se o voo está atrasado ou não, não precisa de *TotalDelayMinutes* (Total de minutos de atraso), *DepDelayMinutes* (Minutos de atraso na partida) e *ArrDelayMinutes* (Minutos de atraso na chegada).

Trate *DepHourofDay* (Hora da partida do dia) como uma variável categórica porque ela não tem nenhuma relação quantitativa com o destino.
- Se você precisasse fazer uma codificação one-hot dessa variável, ela resultaria em mais 23 colunas.
- Outras alternativas para lidar com variáveis categóricas incluem codificação hash, codificação média regularizada e divisão dos valores em buckets, entre outras.
- Neste caso, você só precisa dividir em buckets.

Para alterar um tipo de coluna para categoria, use a função `astype` ([pandas.DataFrame.astype documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.astype.html)).

In [ ]:
data_orig = data.copy()
data = data[[ 'is_delay', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay']]
categorical_columns  = ['Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'DepHourofDay']
for c in categorical_columns:
    data[c] = data[c].astype('category')

Para usar a codificação one-hot, use a função `get_dummies` no pandas para as colunas categóricas que você selecionou. Em seguida, você pode concatenar esses recursos gerados com seu conjunto de dados original usando a função `concat` no pandas. Para codificar variáveis categóricas, você também pode usar *dummy encoding* (codificação fictícia) usando uma palavra-chave `drop_first=True`. Para obter mais informações sobre codificação fictícia, consulte [Variável fictícia (estatísticas)] (https://en.wikiversity.org/wiki/Dummy_variable_(statistics)).

Por exemplo:
```
pd.get_dummies(df[['column1','columns2']], drop_first=True)
```

In [ ]:
data_dummies = pd.get_dummies(<CODE>, drop_first=True) # Enter your code here
data = pd.concat([<CODE>, <CODE>], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

Verifique o tamanho do conjunto de dados e das novas colunas.

**Dica**: Use as propriedades `shape` e `columns`.

In [ ]:
# Enter your code here

In [ ]:
# Enter your code here

Agora, você está pronto para treinar o modelo. Antes de dividir os dados, renomeie a coluna **is_delay** para *target* (destino).

** Dica**: Você pode usar a função `rename` no pandas ([pandas.DataFrame.rename documentation] (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html)).

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

## <span style="color:red"> Fim da etapa 2 </span>

Salve o arquivo do projeto em seu computador local. Siga estas etapas:

1. No explorador de arquivos à esquerda, clique com o botão direito do mouse no bloco de anotações no qual você está trabalhando. 

2. Escolha **Download** (Fazer download) e salve o arquivo localmente.  

Essa ação faz download do bloco de anotações atual para a pasta de download padrão em seu computador.

# Etapa 3: Treinamento e avaliação de modelos

Você deve incluir algumas etapas preliminares ao converter o conjunto de dados de um DataFrame para um formato que um algoritmo de machine learning possa usar. Para o Amazon SageMaker, você deverá executar as seguintes etapas:

1. Divida os dados em `train_data`, `validation_data` e `test_data` usando `sklearn.model_selection.train_test_split`.  

2. Converta o conjunto de dados para um formato de arquivo apropriado que possa ser usado pelo trabalho de treinamento do Amazon SageMaker. Pode ser um arquivo CSV ou um protobuf de registro. Para obter mais informações, consulte [Formatos de dados comuns para treinamento] (https://docs.aws.amazon.com/sagemaker/latest/dg/cdf-training.html).  

3. Faça o upload dos dados no bucket do S3. Se você ainda não tiver criado um, consulte [Criar um bucket] (https://docs.aws.amazon.com/AmazonS3/latest/gsg/CreatingABucket.html).  

Use as células a seguir para concluir essas etapas. Insira e exclua células onde for necessário.

#### <span style="color: blue;">Apresentação do projeto: na apresentação do projeto, anote as principais decisões que você tomou nesta fase.</span>

### Divisão entre treinamento e teste

In [ ]:
from sklearn.model_selection import train_test_split
def split_data(data):
    train, test_and_validate = train_test_split(data, test_size=0.2, random_state=42, stratify=data['target'])
    test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['target'])
    return train, validate, test

In [ ]:
train, validate, test = split_data(data)
print(train['target'].value_counts())
print(test['target'].value_counts())
print(validate['target'].value_counts())

**Exemplo de resposta**
```
0.0 1033570
1.0 274902
Nome: destino, dtype: int64
0.0 129076
1.0 34483
Nome: destino, dtype: int64
0.0 129612
1.0 33947
Nome: destino, dtype: int64
```

### Modelo de classificação de linha de base

In [ ]:
import sagemaker
from sagemaker.serializers import CSVSerializer
from sagemaker.amazon.amazon_estimator import RecordSet
import boto3

# Instantiate the LinearLearner estimator object with 1 ml.m4.xlarge
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=<CODE>,
                                               instance_type=<CODE>,
                                               predictor_type=<CODE>,
                                               binary_classifier_model_selection_criteria=<CODE>)

### Código de exemplo
```
num_classes = len(pd.unique(train_labels))
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                              instance_count=1,
                                              instance_type='ml.m4.xlarge',
                                              predictor_type='binary_classifier',
                                              binary_classifier_model_selection_criteria = 'cross_entropy_loss')
                                              
```

A aprendizagem linear aceita dados de treinamento em tipos de conteúdo protobuf ou CSV. Ela também aceita solicitações de inferência em tipos de conteúdo protobuf, CSV ou JavaScript Object Notation (JSON). Os dados de treinamento têm recursos e rótulos de realidade prática, mas os dados em uma solicitação de inferência têm apenas recursos.

Em um pipeline de produção, a AWS recomenda converter os dados para o formato protobuf do Amazon SageMaker e armazená-los no Amazon S3. Para começar a usar rapidamente, a AWS fornece a operação `record_set` para converter e fazer upload do conjunto de dados quando ele é suficientemente pequeno para caber na memória local. Ele aceita matrizes NumPy, como as que você já tem. Portanto, você o usará para esta etapa. O objeto `RecordSet` rastreará a localização temporária do Amazon S3 dos seus dados. Crie registros de treinamento, validação e teste usando a função `estimator.record_set`. Em seguida, inicie o trabalho de treinamento usando a função `estimator.fit`.

In [ ]:
### Create train, validate, and test records
train_records = classifier_estimator.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

Agora, treine seu modelo no conjunto de dados que você acabou de carregar.

### Código de exemplo
```
linear.fit([train_records,val_records,test_records])
```

In [ ]:
### Fit the classifier
# Enter your code here

## Avaliação do modelo
Nesta seção, você avaliará seu modelo treinado. 

Primeiramente, examine as métricas do trabalho de treinamento:

In [ ]:
sagemaker.analytics.TrainingJobAnalytics(classifier_estimator._current_job_name, 
                                         metric_names = ['test:objective_loss', 
                                                         'test:binary_f_beta',
                                                         'test:precision',
                                                         'test:recall']
                                        ).dataframe()

Em seguida, configure algumas funções que ajudarão a carregar os dados de teste no Amazon S3 e executar uma previsão usando a função de previsão em lote. Usar a previsão em lote ajudará a reduzir custos, pois as instâncias serão executadas somente quando as previsões forem realizadas nos dados de teste fornecidos.

**Observação:** Substitua `<LabBucketName>` pelo nome do bucket de laboratório que foi criado durante a configuração do laboratório.

In [ ]:
import io
#bucket='<LabBucketName>'
prefix='flight-linear'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

In [ ]:
def batch_linear_predict(test_data, estimator):
    batch_X = test_data.iloc[:,1:];
    batch_X_file='batch-in.csv'
    upload_s3_csv(batch_X_file, 'batch-in', batch_X)

    batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
    batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

    classifier_transformer = estimator.transformer(instance_count=1,
                                           instance_type='ml.m4.xlarge',
                                           strategy='MultiRecord',
                                           assemble_with='Line',
                                           output_path=batch_output)

    classifier_transformer.transform(data=batch_input,
                             data_type='S3Prefix',
                             content_type='text/csv',
                             split_type='Line')
    
    classifier_transformer.wait()

    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
    target_predicted_df = pd.read_json(io.BytesIO(obj['Body'].read()),orient="records",lines=True)
    return test_data.iloc[:,0], target_predicted_df.iloc[:,0]


Para executar as previsões no conjunto de dados de teste, execute a função `batch_linear_predict` (que foi definida previamente) no seu conjunto de dados de teste.


In [ ]:
test_labels, target_predicted = batch_linear_predict(test, classifier_estimator)

Para visualizar um gráfico da matriz de confusão e várias métricas de pontuação, crie algumas funções:

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(test_labels, target_predicted):
    matrix = confusion_matrix(test_labels, target_predicted)
    df_confusion = pd.DataFrame(matrix)
    colormap = sns.color_palette("BrBG", 10)
    sns.heatmap(df_confusion, annot=True, fmt='.2f', cbar=None, cmap=colormap)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.ylabel("True Class")
    plt.xlabel("Predicted Class")
    plt.show()
    

In [ ]:
from sklearn import metrics

def plot_roc(test_labels, target_predicted):
    TN, FP, FN, TP = confusion_matrix(test_labels, target_predicted).ravel()
    # Sensitivity, hit rate, recall, or true positive rate
    Sensitivity  = float(TP)/(TP+FN)*100
    # Specificity or true negative rate
    Specificity  = float(TN)/(TN+FP)*100
    # Precision or positive predictive value
    Precision = float(TP)/(TP+FP)*100
    # Negative predictive value
    NPV = float(TN)/(TN+FN)*100
    # Fall out or false positive rate
    FPR = float(FP)/(FP+TN)*100
    # False negative rate
    FNR = float(FN)/(TP+FN)*100
    # False discovery rate
    FDR = float(FP)/(TP+FP)*100
    # Overall accuracy
    ACC = float(TP+TN)/(TP+FP+FN+TN)*100

    print("Sensitivity or TPR: ", Sensitivity, "%") 
    print( "Specificity or TNR: ",Specificity, "%") 
    print("Precision: ",Precision, "%") 
    print("Negative Predictive Value: ",NPV, "%") 
    print( "False Positive Rate: ",FPR,"%")
    print("False Negative Rate: ",FNR, "%") 
    print("False Discovery Rate: ",FDR, "%" )
    print("Accuracy: ",ACC, "%") 

    test_labels = test.iloc[:,0];
    print("Validation AUC", metrics.roc_auc_score(test_labels, target_predicted) )

    fpr, tpr, thresholds = metrics.roc_curve(test_labels, target_predicted)
    roc_auc = metrics.auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % (roc_auc))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic')
    plt.legend(loc="lower right")

    # create the axis of thresholds (scores)
    ax2 = plt.gca().twinx()
    ax2.plot(fpr, thresholds, markeredgecolor='r',linestyle='dashed', color='r')
    ax2.set_ylabel('Threshold',color='r')
    ax2.set_ylim([thresholds[-1],thresholds[0]])
    ax2.set_xlim([fpr[0],fpr[-1]])

    print(plt.figure())

Para plotar a matriz de confusão, chame a função `plot_confusion_matrix` nos dados `test_labels` e `target_predicted` do seu trabalho em lote:

In [ ]:
# Enter your code here

Para imprimir os dados estatísticos e plotar uma curva característica de operação do receptor (ROC), chame a função `plot_roc` no `test_labels` e dados `target_predicted` do trabalho em lote:

In [ ]:
# Enter your code here

### Perguntas importantes a considerar:

1. Como o desempenho do modelo do conjunto de testes se compara ao seu desempenho no conjunto de treinamento? O que você pode deduzir dessa comparação? 
2. Existem diferenças óbvias entre os resultados de métricas como exatidão, precisão e recall? Se esse for o caso, por que você pode observar essas diferenças? 
3. Considerando a situação e as metas do seu negócio, que métricas são as mais importantes para você considerar? Por quê?
4. Do ponto de vista do negócio, o resultado da métrica (ou métricas) que você considera é o mais importante e suficiente para as suas necessidades? Caso contrário, quais algumas das coisas que você pode mudar na próxima iteração? (Isso acontecerá na seção de engenharia de recursos, que virá a seguir.)

Use as células a seguir para responder a essas (e a outras) perguntas. Insira e exclua células onde for necessário.

#### <span style="color: blue;">Apresentação do projeto: na apresentação do projeto, anote as respostas a essas perguntas (e a outras perguntas semelhantes que você pode responder) nesta seção. Registre os principais detalhes e decisões que você tomou.</span>


**Pergunta**: O que você pode resumir com base na matriz de confusão?


In [ ]:
# Enter your answer here

## <span style="color:red"> Fim da Etapa 3 </span>

Salve o arquivo do projeto em seu computador local. Siga estas etapas:

1. No explorador de arquivos à esquerda, clique com o botão direito do mouse no bloco de anotações no qual você está trabalhando. 

2. Selecione **Download** (Fazer download) e salve o arquivo localmente.  

Essa ação faz download do bloco de anotações atual para a pasta de download padrão em seu computador.

# Iteração II

# Etapa 4: Engenharia de recursos

Você passou por uma iteração de treinamento e avaliação do modelo. Considerando que o primeiro resultado que você alcançou para seu modelo provavelmente não foi suficiente para resolver o problema do seu negócio, o que você poderia mudar nos seus dados para possivelmente melhorar o desempenho do modelo?

### Perguntas importantes a considerar:

1. Como o equilíbrio das duas classes principais (*delay* (atraso) e *no delay* (sem atraso)) pode afetar o desempenho do modelo?
2. Você tem alguns recursos correlacionados?
3. Neste estágio, você pode executar alguma técnica de redução de recursos que possa ter um impacto positivo no desempenho do modelo? 
4. Você pode pensar em adicionar mais dados ou conjuntos de dados?
5. Depois de executar alguma engenharia de recursos, como o desempenho do seu modelo se compara com a primeira iteração?

Use as células a seguir para executar técnicas específicas de engenharia de recursos que você acha que podem melhorar o desempenho do modelo (use as perguntas anteriores como guia). Insira e exclua células onde for necessário.

#### <span style="color: blue;">Apresentação do projeto: na apresentação do projeto, registre suas principais decisões e os métodos que você usa nesta seção. Inclua também todas as novas métricas de desempenho obtidas depois de reavaliar o modelo.</span>

Antes de começar, pense no motivo de a precisão e o recall serem cerca de 80% e a exatidão ser 99%.

Adicione mais recursos:

1. Férias
2. Condições meteorológicas

Como a lista de feriados de 2014 a 2018 é conhecida, você pode criar uma variável indicadora **is_holiday** para marcá-los.

A hipótese é que os atrasos nos voos podem ser maiores durante feriados, em comparação com o restante dos dias. Adicione uma variável booleana `is_holiday` que inclua os feriados dos anos de 2014 a 2018.

In [ ]:
# Source: http://www.calendarpedia.com/holidays/federal-holidays-2014.html

holidays_14 = ['2014-01-01',  '2014-01-20', '2014-02-17', '2014-05-26', '2014-07-04', '2014-09-01', '2014-10-13', '2014-11-11', '2014-11-27', '2014-12-25' ] 
holidays_15 = ['2015-01-01',  '2015-01-19', '2015-02-16', '2015-05-25', '2015-06-03', '2015-07-04', '2015-09-07', '2015-10-12', '2015-11-11', '2015-11-26', '2015-12-25'] 
holidays_16 = ['2016-01-01',  '2016-01-18', '2016-02-15', '2016-05-30', '2016-07-04', '2016-09-05', '2016-10-10', '2016-11-11', '2016-11-24', '2016-12-25', '2016-12-26']
holidays_17 = ['2017-01-02', '2017-01-16', '2017-02-20', '2017-05-29' , '2017-07-04', '2017-09-04' ,'2017-10-09', '2017-11-10', '2017-11-23', '2017-12-25']
holidays_18 = ['2018-01-01', '2018-01-15', '2018-02-19', '2018-05-28' , '2018-07-04', '2018-09-03' ,'2018-10-08', '2018-11-12','2018-11-22', '2018-12-25']
holidays = holidays_14+ holidays_15+ holidays_16 + holidays_17+ holidays_18

### Add indicator variable for holidays
data_orig['is_holiday'] = # Enter your code here 

Os dados meteorológicos foram obtidos em https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&amp;stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&amp;dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&amp;startDate=2014-01-01&amp;endDate=2018-12-31.
<br>

Esse conjunto de dados tem informações sobre velocidade do vento, precipitação, neve e temperatura para as cidades, de acordo com os códigos dos seus aeroportos.

**Pergunta**: O mau tempo decorrente de chuva, ventos pesados ou neve pode causar atrasos nos voos? Você verificará agora.

In [ ]:
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMLFO-1/flight_delay_project/data2/daily-summaries.csv /home/ec2-user/SageMaker/project/data/
#!wget 'https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00023174,USW00012960,USW00003017,USW00094846,USW00013874,USW00023234,USW00003927,USW00023183,USW00013881&dataTypes=AWND,PRCP,SNOW,SNWD,TAVG,TMIN,TMAX&startDate=2014-01-01&endDate=2018-12-31' -O /home/ec2-user/SageMaker/project/data/daily-summaries.csv

Importe os dados meteorológicos que foram preparados para os códigos de aeroporto no conjunto de dados. Use as estações e aeroportos a seguir para a análise. Crie uma nova coluna denominada *airport* (aeroporto) que mapeie a estação meteorológica para o nome do aeroporto.

In [ ]:
weather = pd.read_csv('/home/ec2-user/SageMaker/project/data/daily-summaries.csv')
station = ['USW00023174','USW00012960','USW00003017','USW00094846','USW00013874','USW00023234','USW00003927','USW00023183','USW00013881'] 
airports = ['LAX', 'IAH', 'DEN', 'ORD', 'ATL', 'SFO', 'DFW', 'PHX', 'CLT']

### Map weather stations to airport code
station_map = {s:a for s,a in zip(station, airports)}
weather['airport'] = weather['STATION'].map(station_map)

Na coluna **DATE** (DATA), crie outra coluna denominada *MONTH* (MÊS).

In [ ]:
weather['MONTH'] = weather['DATE'].apply(lambda x: x.split('-')[1])
weather.head()

### Exemplo de resultado
```
  STATION     DATE      AWND PRCP SNOW SNWD TAVG TMAX  TMIN airport MONTH
0 USW00023174 2014-01-01 16   0   NaN  NaN 131.0 178.0 78.0  LAX    01
1 USW00023174 2014-01-02 22   0   NaN  NaN 159.0 256.0 100.0 LAX    01
2 USW00023174 2014-01-03 17   0   NaN  NaN 140.0 178.0 83.0  LAX    01
3 USW00023174 2014-01-04 18   0   NaN  NaN 136.0 183.0 100.0 LAX    01
4 USW00023174 2014-01-05 18   0   NaN  NaN 151.0 244.0 83.0  LAX    01
```

Analise e manipule as colunas **SNOW** (NEVE) e **SNWD** (PROFUNDIDADE DA NEVE) para valores ausentes usando `fillna()`. Para verificar os valores ausentes de todas as colunas, use a função `isna()`.

In [ ]:
weather.SNOW.fillna(0, inplace=True)
weather.SNWD.fillna(0, inplace=True)
weather.isna().sum()

**Pergunta**: Imprima o índice das linhas que têm valores ausentes para *TAVG* (TEMPERATURA MÉDIA ), *TMAX* (TEMPERATURA MÁXIMA), *TMIN* (TEMPERATURA MÍNIMA).

**Dica**: para encontrar as linhas ausentes, use a função`isna()`. Em seguida, para obter o índice, use a lista na variável *idx*.

In [ ]:
idx = np.array([i for i in range(len(weather))])
TAVG_idx = idx[weather.TAVG.isna()] 
TMAX_idx = # Enter your code here 
TMIN_idx = # Enter your code here 
TAVG_idx

### Exemplo de resultado

```
array([ 3956,  3957,  3958,  3959,  3960,  3961,  3962,  3963,  3964,
        3965,  3966,  3967,  3968,  3969,  3970,  3971,  3972,  3973,
        3974,  3975,  3976,  3977,  3978,  3979,  3980,  3981,  3982,
        3983,  3984,  3985,  4017,  4018,  4019,  4020,  4021,  4022,
        4023,  4024,  4025,  4026,  4027,  4028,  4029,  4030,  4031,
        4032,  4033,  4034,  4035,  4036,  4037,  4038,  4039,  4040,
        4041,  4042,  4043,  4044,  4045,  4046,  4047, 13420])
```

Você pode substituir os valores ausentes *TAVG* (TEMPERATURA MÉDIA ), *TMAX* (TEMPERATURA MÁXIMA), *TMIN* (TEMPERATURA MÍNIMA) pelo valor médio de uma estação ou aeroporto específico. Como linhas consecutivas de *TAVG_idx* estão ausentes, substituí-las por um valor anterior não será possível. Em vez disso, substitua-as pela média. Use a função `groupby` para agregar as variáveis com um valor médio.

**Dica:** Agrupar por `MONTH` e `STATION`.

In [ ]:
weather_impute = weather.groupby([<CODE>]).agg({'TAVG':'mean','TMAX':'mean', 'TMIN':'mean' }).reset_index()# Enter your code here
weather_impute.head(2)

Mescle os dados médios com os dados meteorológicos.

In [ ]:

weather = pd.merge(weather, weather_impute,  how='left', left_on=['MONTH','STATION'], right_on = ['MONTH','STATION'])\
.rename(columns = {'TAVG_y':'TAVG_AVG',
                   'TMAX_y':'TMAX_AVG', 
                   'TMIN_y':'TMIN_AVG',
                   'TAVG_x':'TAVG',
                   'TMAX_x':'TMAX', 
                   'TMIN_x':'TMIN'})

Verifique novamente se há valores ausentes.

In [ ]:
weather.TAVG[TAVG_idx] = weather.TAVG_AVG[TAVG_idx]
weather.TMAX[TMAX_idx] = weather.TMAX_AVG[TMAX_idx]
weather.TMIN[TMIN_idx] = weather.TMIN_AVG[TMIN_idx]
weather.isna().sum()

Descarte `STATION,MONTH,TAVG_AVG,TMAX_AVG,TMIN_AVG,TMAX,TMIN,SNWD` do conjunto de dados.

In [ ]:
weather.drop(columns=['STATION','MONTH','TAVG_AVG', 'TMAX_AVG', 'TMIN_AVG', 'TMAX' ,'TMIN', 'SNWD'],inplace=True)

Adicione as condições meteorológicas de origem e destino ao conjunto de dados.

In [ ]:
### Add origin weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Origin'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_O','PRCP':'PRCP_O', 'TAVG':'TAVG_O', 'SNOW': 'SNOW_O'})\
.drop(columns=['DATE','airport'])

### Add destination weather conditions
data_orig = pd.merge(data_orig, weather,  how='left', left_on=['FlightDate','Dest'], right_on = ['DATE','airport'])\
.rename(columns = {'AWND':'AWND_D','PRCP':'PRCP_D', 'TAVG':'TAVG_D', 'SNOW': 'SNOW_D'})\
.drop(columns=['DATE','airport'])

**Observação**: É sempre uma boa prática verificar nulos ou NAs após junções.

In [ ]:
sum(data.isna().any())

In [ ]:
data_orig.columns

Converta os dados categóricos em dados numéricos usando codificação one-hot.

In [ ]:
data = data_orig.copy()
data = data[['is_delay', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest','Distance','DepHourofDay','is_holiday', 'AWND_O', 'PRCP_O',
       'TAVG_O', 'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D']]


categorical_columns  = ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 
       'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']
for c in categorical_columns:
    data[c] = data[c].astype('category')

In [ ]:
data_dummies = pd.get_dummies(data[['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'Reporting_Airline', 'Origin', 'Dest', 'is_holiday']], drop_first=True)
data = pd.concat([data, data_dummies], axis = 1)
data.drop(categorical_columns,axis=1, inplace=True)

Verifique as novas colunas.

In [ ]:
data.shape

In [ ]:
data.columns

### Exemplo de resultado

```
Index(['Distance', 'DepHourofDay', 'is_delay', 'AWND_O', 'PRCP_O', 'TAVG_O',
       'AWND_D', 'PRCP_D', 'TAVG_D', 'SNOW_O', 'SNOW_D', 'Year_2015',
       'Year_2016', 'Year_2017', 'Year_2018', 'Quarter_2', 'Quarter_3',
       'Quarter_4', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6',
       'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12',
       'DayofMonth_2', 'DayofMonth_3', 'DayofMonth_4', 'DayofMonth_5',
       'DayofMonth_6', 'DayofMonth_7', 'DayofMonth_8', 'DayofMonth_9',
       'DayofMonth_10', 'DayofMonth_11', 'DayofMonth_12', 'DayofMonth_13',
       'DayofMonth_14', 'DayofMonth_15', 'DayofMonth_16', 'DayofMonth_17',
       'DayofMonth_18', 'DayofMonth_19', 'DayofMonth_20', 'DayofMonth_21',
       'DayofMonth_22', 'DayofMonth_23', 'DayofMonth_24', 'DayofMonth_25',
       'DayofMonth_26', 'DayofMonth_27', 'DayofMonth_28', 'DayofMonth_29',
       'DayofMonth_30', 'DayofMonth_31', 'DayOfWeek_2', 'DayOfWeek_3',
       'DayOfWeek_4', 'DayOfWeek_5', 'DayOfWeek_6', 'DayOfWeek_7',
       'Reporting_Airline_DL', 'Reporting_Airline_OO', 'Reporting_Airline_UA',
       'Reporting_Airline_WN', 'Origin_CLT', 'Origin_DEN', 'Origin_DFW',
       'Origin_IAH', 'Origin_LAX', 'Origin_ORD', 'Origin_PHX', 'Origin_SFO',
       'Dest_CLT', 'Dest_DEN', 'Dest_DFW', 'Dest_IAH', 'Dest_LAX', 'Dest_ORD',
       'Dest_PHX', 'Dest_SFO', 'is_holiday_1'],
      dtype='object')
```

Renomeie a coluna **is_delay** para *target* (destino) novamente. Use o mesmo código usado anteriormente.

In [ ]:
data.rename(columns = {<CODE>:<CODE>}, inplace=True )# Enter your code here

Crie os conjuntos de treinamento novamente.

**Dica:** Use a função `split_data` que você definiu (e usou) anteriormente.

In [ ]:
# Enter your code here

### Novo classificador de linha de base

Agora, veja se esses novos recursos adicionam algum poder preditivo ao modelo.

In [ ]:
# Instantiate the LinearLearner estimator object
classifier_estimator2 = # Enter your code here

### Código de exemplo

```
num_classes = len(pd.unique(train_labels)) 
classifier_estimator = sagemaker.LinearLearner(role=sagemaker.get_execution_role(),
                                               instance_count=1,
                                               instance_type='ml.m4.xlarge',
                                               predictor_type='binary_classifier',
                                               binary_classifier_model_selection_criteria = 'cross_entropy_loss')
```

In [ ]:
train_records = classifier_estimator2.record_set(train.values[:, 1:].astype(np.float32), train.values[:, 0].astype(np.float32), channel='train')
val_records = classifier_estimator2.record_set(validate.values[:, 1:].astype(np.float32), validate.values[:, 0].astype(np.float32), channel='validation')
test_records = classifier_estimator2.record_set(test.values[:, 1:].astype(np.float32), test.values[:, 0].astype(np.float32), channel='test')

Treine seu modelo usando os três conjuntos de dados que você acabou de criar.

In [ ]:
# Enter your code here

Execute uma previsão em lote usando o modelo recém-treinado.

In [ ]:
# Enter your code here

Trace uma matriz de confusão.

In [ ]:
# Enter your code here

Trace a curva ROC.

In [ ]:
# Enter your code here

O modelo linear mostra apenas uma pequena melhoria no desempenho. Experimente um modelo de conjunto baseado em árvore, denominado *XGBoost*, com o Amazon SageMaker.

## Experimentar o modelo XGBoost

Execute as seguintes etapas:  

1. Use as variáveis do conjunto de treinamento e salve-as como arquivos CSV: train.csv, validation.csv e test.csv.
2. Armazene o nome do bucket na variável. O nome do bucket do Amazon S3 é fornecido à esquerda das instruções do laboratório.  
a. `bucket = <LabBucketName>`  
b. `prefix = 'flight-xgb'`  
3. Use o AWS SDK para Python (Boto3) para fazer upload do modelo no bucket.    

In [ ]:
bucket='c218151a5506212l17131779t1w715054373660-labbucket-ntokzfktykpe'
prefix='flight-xgb'
train_file='flight_train.csv'
test_file='flight_test.csv'
validate_file='flight_validate.csv'
whole_file='flight.csv'
s3_resource = boto3.Session().resource('s3')

def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False )
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

Use a função `sagemaker.inputs.TrainingInput` para criar um `record_set` para os conjuntos de dados de treinamento e validação.

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
from sagemaker.image_uris import retrieve
container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
sess = sagemaker.Session()
s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

xgb = sagemaker.estimator.Estimator(container,
                                    role = sagemaker.get_execution_role(), 
                                    instance_count=1, 
                                    instance_type=instance_type,
                                    output_path=s3_output_location,
                                    sagemaker_session=sess)
xgb.set_hyperparameters(max_depth=5,
                        eta=0.2,
                        gamma=4,
                        min_child_weight=6,
                        subsample=0.8,
                        silent=0,
                        objective='binary:logistic',
                        eval_metric = "auc", 
                        num_round=100)

xgb.fit(inputs=data_channels)

Use o transformador em lote para seu novo modelo e avalie o modelo no conjunto de dados de teste.

In [ ]:
batch_X = test.iloc[:,1:];
batch_X_file='batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

Obtenha o destino e os rótulos de teste previstos.

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

Calcule os valores previstos com base no limite definido.

**Observação:** O destino previsto será uma pontuação, que deverá ser convertida em uma classe binária.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

Trace uma matriz de confusão para `target_predicted` e `test_labels`.

In [ ]:
# Enter your code here

Trace o gráfico ROC:

In [ ]:
# Enter your code here

### Experimentar limites diferentes

**Pergunta**: Com base em até que ponto o modelo processou o conjunto de testes, o que você pode concluir?

In [ ]:
#Enter your answer here

### Otimização de hiperparâmetros (HPO)

In [ ]:
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner

### You can spin up multiple instances to do hyperparameter optimization in parallel

xgb = sagemaker.estimator.Estimator(container,
                                    role=sagemaker.get_execution_role(), 
                                    instance_count= 1, # make sure you have a limit set for these instances
                                    instance_type=instance_type, 
                                    output_path='s3://{}/{}/output'.format(bucket, prefix),
                                    sagemaker_session=sess)

xgb.set_hyperparameters(eval_metric='auc',
                        objective='binary:logistic',
                        num_round=100,
                        rate_drop=0.3,
                        tweedie_variance_power=1.4)

hyperparameter_ranges = {'alpha': ContinuousParameter(0, 1000, scaling_type='Linear'),
                         'eta': ContinuousParameter(0.1, 0.5, scaling_type='Linear'),
                         'min_child_weight': ContinuousParameter(3, 10, scaling_type='Linear'),
                         'subsample': ContinuousParameter(0.5, 1),
                         'num_round': IntegerParameter(10,150)}

objective_metric_name = 'validation:auc'

tuner = HyperparameterTuner(xgb,
                            objective_metric_name,
                            hyperparameter_ranges,
                            max_jobs=10, # Set this to 10 or above depending upon budget and available time.
                            max_parallel_jobs=1)

In [ ]:
tuner.fit(inputs=data_channels)
tuner.wait()

<i class="fas fa-exclamation-triangle" style="color:red"></i> Aguarde até que o trabalho de treinamento esteja concluído. Pode demorar 25 a 30 minutos.

**Para monitorar trabalhos de otimização de hiperparâmetros:**  

1. No Console de gerenciamento da AWS, no menu **Services** (Serviços), escolha **Amazon SageMaker**.  
2. Escolha **Training > Hyperparameter tuning jobs** (Treinamento > Trabalhos de ajuste de hiperparâmetros).
3. Você pode verificar o status de cada trabalho de ajuste de hiperparâmetros, seu valor de métrica objetiva e seus logs.  

Verifique se o trabalho foi concluído com êxito.

In [ ]:
boto3.client('sagemaker').describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name)['HyperParameterTuningJobStatus']

O trabalho de ajuste de hiperparâmetros terá um modelo que funcionou melhor. Você pode obter as informações sobre esse modelo no trabalho de ajuste.

In [ ]:
sage_client = boto3.Session().client('sagemaker')
tuning_job_name = tuner.latest_tuning_job.job_name
print(f'tuning job name:{tuning_job_name}')
tuning_job_result = sage_client.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)
best_training_job = tuning_job_result['BestTrainingJob']
best_training_job_name = best_training_job['TrainingJobName']
print(f"best training job: {best_training_job_name}")

best_estimator = tuner.best_estimator()

tuner_df = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_name).dataframe()
tuner_df.head()

Use o estimador `best_estimator` e treine-o usando os dados. 

**Dica:** Veja a função anterior de ajuste do estimador XGBoost.

In [ ]:
# Enter your code here'

Use o transformador em lote para seu novo modelo e avalie o modelo no conjunto de dados de teste.

In [ ]:
batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)
batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = best_estimator.transformer(instance_count=1,
                                       instance_type=instance_type,
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')
xgb_transformer.wait()

In [ ]:
s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),',',names=['target'])
test_labels = test.iloc[:,0]

Obtenha o destino e os rótulos de teste previstos.

In [ ]:
print(target_predicted.head())

def binary_convert(x):
    threshold = 0.55
    if x > threshold:
        return 1
    else:
        return 0

target_predicted['target'] = target_predicted['target'].apply(binary_convert)

test_labels = test.iloc[:,0]

print(target_predicted.head())

Trace uma matriz de confusão para `target_predicted` e `test_labels`.

In [ ]:
# Enter your code here

Trace o gráfico ROC:

In [ ]:
# Enter your code here

**Pergunta**: Tente diferentes hiperparâmetros e intervalos de hiperparâmetros. Essas alterações melhoram o modelo?

## Conclusão

Você já iterou algumas vezes por meio do treinamento e da avaliação do modelo. Está na hora de encerrar esse projeto e refletir sobre:

- O que você aprendeu 
- Que tipos de etapas você pode seguir para avançar (supondo que você tivesse mais tempo)

Use a célula a seguir para responder a algumas dessas perguntas e a outras perguntas relevantes:

1. O desempenho do modelo atende à meta da sua empresa? Se não atende, cite algumas coisas que você gostaria de fazer de forma diferente se tivesse mais tempo para ajustar.
2. Quanto de melhora seu modelo apresentou quando você fez mudanças no seu conjunto de dados, recursos e hiperparâmetros? Que tipos de técnicas você empregou ao longo deste projeto e quais proporcionaram as maiores melhorias no seu modelo?
3. Cite alguns dos maiores desafios que você encontrou ao longo deste projeto.
4. Você tem alguma dúvida sem resposta sobre aspectos do pipeline que não fizeram sentido para você?
5. Quais foram as três coisas mais importantes que você aprendeu sobre machine learning ao trabalhar neste projeto?

#### <span style="color: blue;">Apresentação do projeto: faça também um resumo das suas respostas a essas perguntas na apresentação do projeto. Combine todas as suas anotações para a apresentação do projeto e prepare-se para apresentar suas descobertas à turma.</span>

In [ ]:
# Write your answers here